# 02 · Feature engineering

Builds the **municipality-level feature table** for the Comunidad de Madrid (179 municipalities)
from the raw files saved by `01_data_collection`. Nothing here goes online except an optional
fallback that fetches a missing boundary.

| Input (`data/raw/`) | Source |
|---|---|
| `boundaries/municipios.gpkg` | OSM admin_level=8 (Geofabrik extract) |
| `ocm/ocm_chargers.csv` | OpenChargeMap |
| `osm/osm_pois.gpkg` | OSM POI layers |
| `ine/ine_population_2881.csv` | INE padrón, table 2881 |
| `ine/ine_adrh_renta_31097.csv` | INE Atlas de Distribución de Renta, table 31097 |
| `dgt/DatosMunicipalesGeneral_2025.xlsx` | DGT municipal vehicle data |

**Outputs:** `data/processed/features_municipal.csv` (for modelling) and `features_municipal.gpkg` (for maps).

Run top to bottom. Every cell is safe to re-run.

## 0 · Setup

In [1]:
import os, re, unicodedata
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import pyogrio

if Path.cwd().name == "notebooks":      # run from the repo root so relative paths work
    os.chdir("..")
print("Working in:", os.getcwd())

CRS = 25830                              # ETRS89 / UTM 30N, metres
RAW, OUT = Path("data/raw"), Path("data/processed")
OUT.mkdir(parents=True, exist_ok=True)

def first_existing(*paths):
    for p in map(Path, paths):
        if p.exists():
            return p
    raise FileNotFoundError("None of these exist:\n  " + "\n  ".join(map(str, paths)))

PATHS = {
    "boundaries": first_existing(RAW / "boundaries/municipios.gpkg"),
    "ocm":        first_existing(RAW / "ocm/ocm_chargers.csv"),
    "osm":        first_existing(RAW / "osm/osm_pois.gpkg"),
    "population": first_existing(RAW / "ine/ine_population_2881.csv", RAW / "ine_population_2881.csv",
                                 RAW / "ine/2881.csv", RAW / "2881.csv"),
    "income":     first_existing(RAW / "ine/ine_adrh_renta_31097.csv", RAW / "ine_adrh_renta_31097.csv",
                                 RAW / "ine/31097.csv", RAW / "31097.csv"),
    "dgt":        first_existing(RAW / "dgt/DatosMunicipalesGeneral_2025.xlsx",
                                 RAW / "DatosMunicipalesGeneral_2025.xlsx"),
}
for k, v in PATHS.items():
    print(f"  {k:<11} {v}")

Working in: e:\CodingProjects\ChargingStation
  boundaries  data\raw\boundaries\municipios.gpkg
  ocm         data\raw\ocm\ocm_chargers.csv
  osm         data\raw\osm\osm_pois.gpkg
  population  data\raw\2881.csv
  income      data\raw\ine\ine_adrh_renta_31097.csv
  dgt         data\raw\dgt\DatosMunicipalesGeneral_2025.xlsx


In [2]:
def ine_num(s):
    """INE text numbers -> floats: '3.332.035' -> 3332035, '16.523,5' -> 16523.5, '..' -> NaN."""
    s = s.astype(str).str.strip()
    return pd.to_numeric(s.str.replace(".", "", regex=False).str.replace(",", ".", regex=False),
                         errors="coerce")

def norm(s):
    """Normalise a municipality name so INE, OSM and DGT spellings match.
    'Rozas de Madrid, Las' and 'Las Rozas de Madrid' -> 'las rozas de madrid'."""
    s = str(s).strip()
    m = re.match(r"^(.*),\s*(La|El|Los|Las)$", s, flags=re.I)
    if m:
        s = f"{m.group(2)} {m.group(1)}"
    s = unicodedata.normalize("NFKD", s).encode("ascii", "ignore").decode().lower()
    return re.sub(r"\s+", " ", re.sub(r"[^a-z ]", " ", s)).strip()

## 1 · Population (INE padrón) — the master list of 179 municipalities

In [3]:
pop = pd.read_csv(PATHS["population"], sep=";", encoding="utf-8-sig", dtype=str)
pop = pop[(pop["Sexo"] == "Total") & pop["Municipios"].str.match(r"^\d{5} ", na=False)]
POP_YEAR = pop["Periodo"].max()
pop = pop[pop["Periodo"] == POP_YEAR].copy()
pop["ine_code"] = pop["Municipios"].str[:5]
pop["ine_name"] = pop["Municipios"].str[6:]
pop["population"] = ine_num(pop["Total"])
pop = pop[["ine_code", "ine_name", "population"]].drop_duplicates("ine_code").reset_index(drop=True)

print(f"Population year {POP_YEAR}: {len(pop)} municipalities, "
      f"{pop['population'].sum():,.0f} residents")

Population year 2025: 179 municipalities, 7,137,031 residents


## 2 · Boundaries

Keeps one polygon per INE code. If a municipality is missing from the OSM extract,
it fetches that boundary from Nominatim (the one online step in this notebook) and saves it back.

In [4]:
FETCH_MISSING = True   # set False to skip the online fallback

munis = gpd.read_file(PATHS["boundaries"]).to_crs(CRS)
munis["ine_code"] = munis["ine_code"].astype(str).str.extract(r"(\d{5})", expand=False)

# Codes that are missing or unknown: try to recover them from the name
valid = set(pop["ine_code"])
name_to_code = dict(zip(pop["ine_name"].map(norm), pop["ine_code"]))
bad = ~munis["ine_code"].isin(valid)
munis.loc[bad, "ine_code"] = munis.loc[bad, "name"].map(norm).map(name_to_code)
if bad.any():
    print("Recovered by name:", munis.loc[bad & munis["ine_code"].notna(), "name"].tolist())
    print("Dropped (not municipalities):", munis.loc[munis["ine_code"].isna(), "name"].tolist())
munis = munis[munis["ine_code"].notna()]

# One row per municipality (merges any split polygons)
munis = munis.dissolve(by="ine_code", as_index=False, aggfunc="first")[["ine_code", "name", "geometry"]]

missing = pop[~pop["ine_code"].isin(munis["ine_code"])]
if len(missing) and FETCH_MISSING:
    try:
        import osmnx as ox
    except ImportError:
        ox = None
        print("osmnx not installed; cannot fetch missing boundaries")
    added = []
    for code, ine_name in (zip(missing["ine_code"], missing["ine_name"]) if ox else []):
        name = re.sub(r"^(.*),\s*(La|El|Los|Las)$", r"\2 \1", ine_name)
        try:
            g = ox.geocode_to_gdf(f"{name}, Comunidad de Madrid, España").to_crs(CRS)
        except Exception as e:
            print(f"  could not fetch {name}: {e}")
            continue
        if g.geom_type.iloc[0] not in ("Polygon", "MultiPolygon"):
            print(f"  {name}: Nominatim returned a {g.geom_type.iloc[0]}, not a boundary; skipped")
            continue
        added.append(gpd.GeoDataFrame({"ine_code": [code], "name": [name]},
                                      geometry=[g.geometry.iloc[0]], crs=CRS))
        print(f"  fetched {code} {name}: {g.area.iloc[0] / 1e6:.1f} km²")
    if added:
        munis = pd.concat([munis, *added], ignore_index=True)
        munis.to_file(PATHS["boundaries"])
        print("Saved updated boundaries to", PATHS["boundaries"])

munis["name"] = munis["name"].fillna(munis["ine_code"].map(dict(zip(pop["ine_code"], pop["ine_name"]))))
munis = munis.sort_values("ine_code").reset_index(drop=True)

still_missing = pop.loc[~pop["ine_code"].isin(munis["ine_code"]), "ine_name"].tolist()
print(f"{len(munis)} municipalities with boundaries (target {len(pop)}). Missing: {still_missing or 'none'}")

  fetched 28135 Santa María de la Alameda: 75.4 km²
Saved updated boundaries to data\raw\boundaries\municipios.gpkg
179 municipalities with boundaries (target 179). Missing: none


## 3 · Income (INE Atlas de Distribución de Renta de los Hogares)

In [5]:
adrh = pd.read_csv(PATHS["income"], sep=";", encoding="utf-8-sig", dtype=str)
print("Columns:", adrh.columns.tolist())

sub_geo = [c for c in ("Distritos", "Secciones") if c in adrh.columns]
mun = adrh[adrh[sub_geo].isna().all(axis=1)] if sub_geo else adrh       # municipality rows only
mun = mun[mun["Municipios"].str.match(r"^\d{5} ", na=False)].copy()

ind_col = next(c for c in mun.columns if "renta" in c.lower())
INCOME_YEAR = mun["Periodo"].max()
mun = mun[mun["Periodo"] == INCOME_YEAR]
mun["ine_code"] = mun["Municipios"].str[:5]
mun["value"] = ine_num(mun["Total"])

income = mun.pivot_table(index="ine_code", columns=ind_col, values="value")
INCOME_COLS = {"Renta neta media por persona": "income_net_pp",
               "Renta neta media por hogar": "income_net_hh",
               "Mediana de la renta por unidad de consumo": "income_median_uc"}
income = income[[c for c in INCOME_COLS if c in income.columns]].rename(columns=INCOME_COLS)

print(f"Income year {INCOME_YEAR}: {len(income)} municipalities; missing values: "
      f"{income.isna().sum().to_dict()}")

Columns: ['Municipios', 'Distritos', 'Secciones', 'Indicadores de renta media y mediana', 'Periodo', 'Total']
Income year 2023: 179 municipalities; missing values: {'income_net_pp': 0, 'income_net_hh': 0, 'income_median_uc': 6}


## 4 · Vehicles (DGT) — EV demand

**EV definition:** vehicles with the DGT **CERO** environmental label (battery EVs, plus plug-in
hybrids with ≥40 km electric range).

The first cell shows the layout of every sheet. The second tries to find the right sheet and
columns automatically and prints what it picked. If it picks wrong, set the `DGT_*` overrides.

In [6]:
xl = pd.ExcelFile(PATHS["dgt"])
print("Sheets:", xl.sheet_names)
for s in xl.sheet_names:
    print(f"\n=== {s}")
    print(xl.parse(s, header=None, nrows=8).iloc[:, :14].to_string())

Sheets: ['DatosMunicipalesGeneral_2025']

=== DatosMunicipalesGeneral_2025
           0                                        1            2                   3                4                  5                  6                    7                    8                  9                    10                   11               12                 13
0  Código INE                                Municipio    Provincia  Comunidad Autónoma  Población Total  Población Hombres  Población Mujeres  Conductores Hombres  Conductoras Mujeres  Censo Conductores  Parque Ciclomotores  Parque Motocicletas  Parque Turismos  Parque Furgonetas
1       01000  Araba/Álava (Municipio Sin Especificar)  Araba/Álava          País Vasco              NaN                NaN                NaN                  NaN                  NaN                NaN                    0                    0                9                  0
2       01001                         Alegría-Dulantzi  Araba/Álava          Pa

In [14]:
LABEL_COLS = ["Distintivo 0", "Distintivo ECO", "Distintivo C", "Distintivo B", "Sin Distintivo"]

dgt = xl.parse(xl.sheet_names[0], header=0)
dgt.columns = [str(c).strip() for c in dgt.columns]
print("First columns:", list(dgt.columns[:8]))

def norm2(s):
    s = re.sub(r"^(.*?)\s*\((La|El|Los|Las|L')\)$", r"\2 \1", str(s).strip(), flags=re.I)  # "Rozas de Madrid (Las)"
    return norm(s)

def digits(col, width):
    return pd.to_numeric(dgt[col], errors="coerce").astype("Int64").astype(str).str.zfill(width)

cols = list(dgt.columns)
c_ine  = next((c for c in cols if re.search(r"\bINE\b", c, re.I)), None)
c_prov = next((c for c in cols if re.search(r"c[oó]d.*prov", c, re.I)), None)
c_mun  = next((c for c in cols if re.search(r"c[oó]d.*mun", c, re.I)), None)
c_pname = next((c for c in cols if re.fullmatch(r"provincia", c, re.I)), None)
c_name = next((c for c in cols if re.search(r"municipio", c, re.I) and not re.search(r"c[oó]d|ine", c, re.I)), None)

dgt["ine_code"] = pd.NA
if c_ine:                                    # one column with the 5-digit INE code
    dgt["ine_code"] = digits(c_ine, 5)
if c_prov and c_mun and not dgt["ine_code"].isin(valid).any():   # province + municipality codes
    dgt["ine_code"] = digits(c_prov, 2) + digits(c_mun, 3)
if not dgt["ine_code"].isin(valid).any():    # fall back to names, Madrid province only
    madrid = dgt
    if c_pname:
        madrid = dgt[dgt[c_pname].astype(str).str.contains("madrid", case=False)]
    elif c_prov:
        madrid = dgt[digits(c_prov, 2) == "28"]
    name_map = {**name_to_code, **{norm2(n): c for n, c in zip(pop["ine_name"], pop["ine_code"])}}
    dgt.loc[madrid.index, "ine_code"] = madrid[c_name].map(norm2).map(name_map)
    print("  matched on names")
print(f"  code col: {c_ine!r} | prov/mun cols: {c_prov!r}/{c_mun!r} | name col: {c_name!r}")

dgt = dgt[dgt["ine_code"].isin(valid)].copy()
for c in LABEL_COLS:
    dgt[c] = pd.to_numeric(dgt[c], errors="coerce")
dgt["ev_cero"] = dgt["Distintivo 0"]
dgt["vehicles_total"] = dgt[LABEL_COLS].sum(axis=1, min_count=1)
vehicles = dgt.groupby("ine_code")[["ev_cero", "vehicles_total"]].sum(min_count=1)

unmatched = pop.loc[~pop["ine_code"].isin(vehicles.index), "ine_name"].tolist()
print(f"{len(vehicles)} of {len(pop)} municipalities matched | "
      f"{vehicles['ev_cero'].sum():,.0f} label-0 vehicles of {vehicles['vehicles_total'].sum():,.0f} "
      f"({vehicles['ev_cero'].sum() / vehicles['vehicles_total'].sum():.1%})")
if unmatched:
    print("Not matched:", unmatched)
vehicles.sort_values("ev_cero", ascending=False).head()

First columns: ['Código INE', 'Municipio', 'Provincia', 'Comunidad Autónoma', 'Población Total', 'Población Hombres', 'Población Mujeres', 'Conductores Hombres']
  code col: 'Código INE' | prov/mun cols: None/None | name col: 'Municipio'
179 of 179 municipalities matched | 320,176 label-0 vehicles of 5,523,179 (5.8%)


,ev_cero,vehicles_total
ine_code,,
28079,87717.0,1832369.0
28006,42867.0,455591.0
28080,25232.0,267592.0
28169,14285.0,111913.0
28042,11226.0,88513.0


## 5 · Chargers (OpenChargeMap)

Public = `public` + `public_membership` (membership networks like Iberdrola or Endesa are open to anyone with the app).
Sites with an unknown number of charging points count as 1.

In [8]:
ocm = pd.read_csv(PATHS["ocm"])
ocm["is_public"] = ocm["access_class"].isin(["public", "public_membership"])
ocm["points"] = ocm["n_points"].fillna(1).clip(lower=1)

chg = gpd.GeoDataFrame(ocm, geometry=gpd.points_from_xy(ocm["lon"], ocm["lat"]), crs=4326).to_crs(CRS)
chg = gpd.sjoin(chg, munis[["ine_code", "geometry"]], how="inner", predicate="within").drop(columns="index_right")

pub = chg[chg["is_public"]]
chargers = pd.DataFrame({
    "public_sites":      pub.groupby("ine_code").size(),
    "public_points":     pub.groupby("ine_code")["points"].sum(),
    "public_fast_sites": pub[pub["max_power_kw"] >= 50].groupby("ine_code").size(),   # DC fast
    "private_sites":     chg[~chg["is_public"]].groupby("ine_code").size(),
})

print(f"{len(chg):,} sites inside the Comunidad; {chg['is_public'].mean():.0%} public "
      f"({len(pub):,} sites, {pub['points'].sum():,.0f} points)")
print(f"n_points missing for {ocm['n_points'].isna().mean():.0%} of sites (counted as 1)")
print("\nAccess breakdown (input for the 'share actually public' chart):")
print(chg["access_class"].value_counts().to_string())

2,232 sites inside the Comunidad; 75% public (1,677 sites, 8,195 points)
n_points missing for 0% of sites (counted as 1)

Access breakdown (input for the 'share actually public' chart):
access_class
public_membership    1413
restricted            554
public                264
unknown                 1


## 6 · Points of interest (OSM)

In [9]:
poi = {}
for layer in pyogrio.list_layers(PATHS["osm"])[:, 0]:
    if layer == "charging":          # OCM is the charger source; OSM chargers are only a cross-check
        continue
    g = gpd.read_file(PATHS["osm"], layer=layer).to_crs(CRS)
    j = gpd.sjoin(g[["geometry"]], munis[["ine_code", "geometry"]], how="inner", predicate="within")
    poi[f"n_{layer}"] = j.groupby("ine_code").size()
    print(f"  {layer:<8} {len(g):>7,} features, {len(j):>7,} inside a municipality")
poi = pd.DataFrame(poi).rename(columns={"n_fuel": "fuel_stations"})

  fuel         864 features,     864 inside a municipality
  retail     4,003 features,   4,003 inside a municipality
  office     3,925 features,   3,924 inside a municipality
  food      11,626 features,  11,620 inside a municipality
  transit      492 features,     488 inside a municipality
  parking   14,562 features,  14,541 inside a municipality


## 7 · Accessibility: distance to the nearest public charger

Straight-line distance (km) from each municipality's representative point to the nearest public charger,
in any municipality. A coarse first version; road-network distance is a Tier 3 upgrade.

In [10]:
centres = munis[["ine_code"]].copy()
centres["geometry"] = munis.geometry.representative_point()
centres = gpd.GeoDataFrame(centres, geometry="geometry", crs=CRS)

nearest = gpd.sjoin_nearest(centres, pub[["geometry"]], how="left", distance_col="dist_m")
access = (nearest.groupby("ine_code")["dist_m"].min() / 1000).rename("dist_nearest_public_km").to_frame()
access.describe().T.round(2)

,count,mean,std,min,25%,50%,75%,max
dist_nearest_public_km,179.0,3.15,2.84,0.05,0.82,2.33,4.73,12.16


## 8 · Assemble the feature table

In [16]:
feat = munis.copy()
feat["area_km2"] = feat.area / 1e6

for table in (pop.set_index("ine_code")[["population"]], income, vehicles, chargers, poi, access):
    feat = feat.drop(columns=[c for c in table.columns if c in feat.columns])
    feat = feat.merge(table, left_on="ine_code", right_index=True, how="left")

# Counts: a municipality with no match genuinely has zero
count_cols = list(chargers.columns) + list(poi.columns)
feat[count_cols] = feat[count_cols].fillna(0).astype(int)

def per(num, den, k):
    return (num / den.replace(0, np.nan)) * k

# Demand side
feat["pop_density_km2"]   = per(feat["population"], feat["area_km2"], 1)          # proxy for apartment living / no garage
feat["ev_share"]          = per(feat["ev_cero"], feat["vehicles_total"], 1)
feat["evs_per_1k_pop"]    = per(feat["ev_cero"], feat["population"], 1_000)
feat["vehicles_per_1k_pop"] = per(feat["vehicles_total"], feat["population"], 1_000)
feat["log_income_net_pp"] = np.log(feat["income_net_pp"]) if "income_net_pp" in feat else np.nan

# Supply side
feat["public_points_per_10k_pop"] = per(feat["public_points"], feat["population"], 10_000)
feat["public_points_per_1k_ev"]   = per(feat["public_points"], feat["ev_cero"], 1_000)
feat["share_public_sites"]        = per(feat["public_sites"], feat["public_sites"] + feat["private_sites"], 1)

# README benchmark: gas stations per 1k combustion vehicles vs chargers per 1k EVs
feat["fuel_per_1k_ice"] = per(feat["fuel_stations"], feat["vehicles_total"] - feat["ev_cero"], 1_000)

# POI intensity per 10k residents
for c in [c for c in poi.columns if c.startswith("n_")]:
    feat[f"{c[2:]}_per_10k_pop"] = per(feat[c], feat["population"], 10_000)

print(feat.shape)
feat.drop(columns="geometry").head()

(179, 35)


,ine_code,name,area_km2,population,income_net_pp,income_net_hh,income_median_uc,ev_cero,vehicles_total,public_sites,...,log_income_net_pp,public_points_per_10k_pop,public_points_per_1k_ev,share_public_sites,fuel_per_1k_ice,retail_per_10k_pop,office_per_10k_pop,food_per_10k_pop,transit_per_10k_pop,parking_per_10k_pop
0,28001,La Acebeda,22.040389,70,16893.0,31213.0,NaN,0.0,70.0,0,...,9.734655,0.000000,NaN,NaN,0.000000,0.000000,0.000000,142.857143,0.000000,0.000000
1,28002,Ajalvir,19.645106,4868,16039.0,45022.0,22050.0,744.0,15400.0,3,...,9.682779,8.216927,5.376344,0.750000,0.204694,12.325390,14.379622,24.650781,0.000000,18.488085
2,28003,Alameda del Valle,25.245638,267,19327.0,34432.0,22750.0,4.0,230.0,0,...,9.869258,0.000000,0.000000,NaN,0.000000,37.453184,0.000000,187.265918,0.000000,37.453184
3,28004,El Álamo,22.167605,10594,12605.0,36817.0,17850.0,168.0,7133.0,2,...,9.441849,3.775722,23.809524,1.000000,0.430725,2.831792,0.000000,3.775722,0.000000,5.663583
4,28005,Alcalá de Henares,87.380559,203208,15598.0,41866.0,20650.0,2815.0,126973.0,56,...,9.654898,12.794772,92.362345,0.727273,0.249682,5.167119,7.922916,12.696351,0.246053,15.107673


In [20]:
# 1. How skewed is it? Most municipalities have well under 1 vehicle per resident
v = feat["vehicles_per_1k_pop"]
print(v.describe(percentiles=[.25, .5, .75, .9]).round(0).to_string())
for cut in (1_000, 1_500, 2_000, 3_000):
    print(f"  > {cut:>5} per 1k residents: {(v > cut).sum():>3} municipalities")

FLEET_CUTOFF = 2 * v.median()          # 'twice the typical rate'; adjust after looking at the printout
feat["fleet_domicile_flag"] = v > FLEET_CUTOFF
print(f"\nCutoff {FLEET_CUTOFF:,.0f} -> {feat['fleet_domicile_flag'].sum()} flagged, "
      f"holding {feat.loc[feat['fleet_domicile_flag'], 'ev_cero'].sum() / feat['ev_cero'].sum():.0%} of all label-0 vehicles")

# 2. Drivers are registered at their home address, so fleets don't distort them
drv = pd.to_numeric(dgt["Censo Conductores"], errors="coerce").groupby(dgt["ine_code"]).sum(min_count=1)
feat["drivers"] = feat["ine_code"].map(drv)
feat["drivers_per_1k_pop"] = per(feat["drivers"], feat["population"], 1_000)
feat["fuel_per_1k_drivers"] = per(feat["fuel_stations"], feat["drivers"], 1_000)

# 3. Resident-EV estimate: learn EVs per resident from unflagged municipalities
#    (log EVs per capita ~ log income + log density, population-weighted), predict for flagged ones
inc = feat["income_net_pp"].fillna(feat["income_net_pp"].median())
X_all = np.column_stack([np.ones(len(feat)), np.log(inc), np.log(feat["pop_density_km2"])])
fit = (~feat["fleet_domicile_flag"]) & (feat["ev_cero"] > 0)
w = np.sqrt(feat.loc[fit, "population"].to_numpy())
y = np.log(feat.loc[fit, "ev_cero"] / feat.loc[fit, "population"]).to_numpy()
beta, *_ = np.linalg.lstsq(X_all[fit.to_numpy()] * w[:, None], y * w, rcond=None)
resid = y - X_all[fit.to_numpy()] @ beta
smear = np.average(np.exp(resid), weights=w ** 2)          # Duan smearing: undo log bias

pred_ev = np.exp(X_all @ beta) * smear * feat["population"]
feat["ev_resident_est"] = np.where(feat["fleet_domicile_flag"], pred_ev, feat["ev_cero"])
feat["evs_per_1k_pop_adj"] = per(feat["ev_resident_est"], feat["population"], 1_000)

print(f"\nEV model: log(EV/pop) = {beta[0]:.2f} + {beta[1]:.2f}·log(income) + {beta[2]:.2f}·log(density)")
print(feat.loc[feat["fleet_domicile_flag"], ["name", "population", "ev_cero", "ev_resident_est", "drivers_per_1k_pop"]]
          .sort_values("ev_cero", ascending=False).head(12).round(0).to_string(index=False))

count       179.0
mean       3677.0
std       16541.0
min         488.0
25%         737.0
50%         830.0
75%         970.0
90%        3030.0
max      176830.0
  >  1000 per 1k residents:  38 municipalities
  >  1500 per 1k residents:  22 municipalities
  >  2000 per 1k residents:  19 municipalities
  >  3000 per 1k residents:  18 municipalities

Cutoff 1,661 -> 19 flagged, holding 45% of all label-0 vehicles

EV model: log(EV/pop) = -26.38 + 2.42·log(income) + -0.14·log(density)
                name  population  ev_cero  ev_resident_est  drivers_per_1k_pop
          Alcobendas      123342  42867.0           4658.0               544.0
         Majadahonda       73625  25232.0           3761.0               625.0
           Venturada        2622  14285.0            150.0               658.0
 Colmenar del Arroyo        2044  11226.0             61.0               542.0
             Patones         617   9785.0             24.0               488.0
         Moralzarzal       14772   9439

## 9 · Quality checks and save

In [17]:
assert feat["ine_code"].is_unique, "duplicate municipalities"
print(f"{len(feat)} municipalities (target {len(pop)})")

miss = feat.drop(columns="geometry").isna().sum()
print("\nMissing values per column:")
print(miss[miss > 0].to_string() if miss.any() else "  none")

print("\nSanity: ten biggest municipalities")
print(feat.nlargest(10, "population")[["name", "population", "ev_cero", "income_net_pp",
                                       "public_points", "public_points_per_10k_pop"]].to_string(index=False))

meta = {"population_year": POP_YEAR, "income_year": INCOME_YEAR, "dgt_file": PATHS["dgt"].name}
print("\nVintages:", meta)

feat.drop(columns="geometry").to_csv(OUT / "features_municipal.csv", index=False)
feat.to_file(OUT / "features_municipal.gpkg", driver="GPKG")
print(f"\nSaved {OUT / 'features_municipal.csv'} and {OUT / 'features_municipal.gpkg'}")

179 municipalities (target 179)

Missing values per column:
income_median_uc            6
public_points_per_1k_ev     3
share_public_sites         69

Sanity: ten biggest municipalities
             name  population  ev_cero  income_net_pp  public_points  public_points_per_10k_pop
           Madrid     3506730  87717.0        19632.0           3664                  10.448480
         Móstoles      214817   2219.0        14045.0            110                   5.120638
Alcalá de Henares      203208   2815.0        15598.0            260                  12.794772
          Leganés      195734   2746.0        14531.0            261                  13.334423
           Getafe      193238   3726.0        15867.0            184                   9.521937
      Fuenlabrada      190076   2027.0        13454.0            139                   7.312864
         Alcorcón      175719   2358.0        15667.0            275                  15.649987
Torrejón de Ardoz      143526   1975.0        

### Data notes for the report
- **Boundaries:** OSM (Geofabrik extract), not the Comunidad portal; any boundary missing from the extract was fetched from Nominatim.
- **Gas stations:** OSM `amenity=fuel` (the README's fallback), not MITECO.
- **Parking:** no municipal "% dwellings without garage" exists for all 179 municipalities; `pop_density_km2` is the proxy.
  INE Census 2021 garage data exists only for municipalities above 50,000 residents, so use it as a robustness check.
- **Small municipalities:** per-capita ratios are unstable for small populations. The regression should use population
  as an exposure/offset rather than per-capita ratios.